# colab_01_label — FREE-ONLY real labels: 8B (cheap) vs 70B (premium-proxy) via Groq

**Cost: $0.** Uses ONE free Groq key for both tiers. No OpenAI.

Input gate: asserts `DECISION_00.json:query_sha256`. Output freeze: `calibration_real.jsonl` + `splits.json` + `DECISION_01.json`.

Manual step at end: 150-row `human_check.csv` for you to mark y/n (~45 min).

In [ ]:
# 0. Setup — self-recovering: finds repo, unzips upload, or clones (public or private via token)
import os, sys, json, hashlib, subprocess, zipfile
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    DRIVE = Path('/content/drive/MyDrive/Research/colab_ckpt')
    DRIVE.mkdir(parents=True, exist_ok=True)
    print('Drive:', DRIVE)
except Exception as e:
    print('No Colab Drive (local run?):', e)
    DRIVE = Path('./colab_ckpt')
    DRIVE.mkdir(parents=True, exist_ok=True)

def _find_backend():
    """Return BACKEND dir (containing src/gateway/service.py) or None."""
    cands = [Path('/content/code-mixed-gateway/backend'), Path('/content/Research/backend'),
             Path('/content/backend'), Path.cwd()/'backend', Path.cwd()/'code-mixed-gateway'/'backend']
    try:
        for p in Path('/content').iterdir():
            if p.is_dir():
                cands += [p, p/'backend']
    except Exception: pass
    for c in cands:
        if (c/'src'/'gateway'/'service.py').exists(): return c
    # deep search one level for upload_tmp style dirs
    try:
        for p in Path('/content').glob('*/**/service.py'):
            if p.parent.parent.name == 'gateway' and p.parent.name == 'gateway':
                be = p.parent.parent.parent  # .../src/gateway/service.py -> backend?
                # walk up to dir containing 'src'
                q = p.parent
                while q != Path('/content') and q.name != 'src': q = q.parent
                be = q.parent
                if (be/'src'/'gateway'/'service.py').exists(): return be
    except Exception: pass
    return None

def _try_unzip_uploads():
    roots = [Path('/content')]
    try: roots.append(DRIVE)
    except Exception: pass
    for root in roots:
        try: zips = list(root.glob('backend*.zip'))
        except Exception: continue
        for z in zips:
            dest = Path('/content/repo_upload')
            if (dest/'backend'/'src'/'gateway'/'service.py').exists(): return True
            print(f'unzipping {z} -> {dest} ...')
            dest.mkdir(parents=True, exist_ok=True)
            with zipfile.ZipFile(z) as zf: zf.extractall(dest)
            # handle zip containing backend/ at top or nested one level
            if not (dest/'backend'/'src'/'gateway'/'service.py').exists():
                for sub in dest.iterdir():
                    if (sub/'backend'/'src'/'gateway'/'service.py').exists():
                        import shutil
                        shutil.move(str(sub/'backend'), str(dest/'backend_tmp'))
                        shutil.move(str(dest/'backend_tmp'), str(dest/'backend'))
                        break
            print('unzip done')
            return True
    return False

def _try_clone():
    dest = Path('/content/code-mixed-gateway')
    if dest.exists(): return dest.exists()
    token = ''
    try:
        from google.colab import userdata
        token = userdata.get('GITHUB_TOKEN') or ''
    except Exception: token = os.environ.get('GITHUB_TOKEN','') or ''
    url = ('https://'+token+'@github.com/tusharsaharan/code-mixed-gateway.git') if token else 'https://github.com/tusharsaharan/code-mixed-gateway.git'
    print('cloning (token=%s) ...' % ('yes' if token else 'no'))
    r = subprocess.run(['git','clone',url,str(dest)], capture_output=True, text=True)
    print('clone rc=', r.returncode)
    if r.returncode != 0: print((r.stderr or '')[-500:])
    return dest.exists()

BACKEND = _find_backend()
if BACKEND is None and _try_unzip_uploads(): BACKEND = _find_backend()
if BACKEND is None and _try_clone(): BACKEND = _find_backend()
print('BACKEND =', BACKEND)
assert BACKEND is not None, ('REPO NOT FOUND. Fastest fix (2 min): on your PC zip the backend folder '
    '(Research/backend -> backend.zip), drag backend.zip into Colab Files panel (folder icon, left), then '
    'Cell > Run this cell again. Alternative: make the GitHub repo public, or add a GITHUB_TOKEN secret.')
REPO = BACKEND.parent
DATA = BACKEND/'data'
print('REPO =', REPO, '| DATA exists:', DATA.exists())

%pip install -q pydantic pydantic-settings numpy httpx 2>&1 | tail -1
src = str(BACKEND/'src')
if src not in sys.path: sys.path.insert(0, src)
import gateway
print('import gateway OK from', gateway.__file__)


In [ ]:
# 1. Build FREE query set: committed seeds + deterministic paraphrases (no API to build queries)
import json
from pathlib import Path
# REPO/BACKEND/DATA already set in Cell 0 (robust finder) — reuse
bench = [json.loads(l) for l in open(DATA/'benchmark.sample.jsonl', encoding='utf-8')]
seeds = [json.loads(l) for l in open(DATA/'seed_hinglish.jsonl', encoding='utf-8')]
humans = [json.loads(l) for l in open(DATA/'seed_human.jsonl', encoding='utf-8')]

FLUFF = ['yaar','matlab','bhai','arre','sun na','dekho na','actually','basically']
rng = random.Random(42)  # FROZEN seed — do not change or splits hash breaks
queries = []
for r in bench:
    queries.append({'id': r['id'], 'text': r['original'], 'ref': r['reference_answer'], 'src': 'bench'})
for r in seeds:
    queries.append({'id': r['id'], 'text': r['text'], 'ref': r['text'], 'src': 'seed'})
for r in humans:
    queries.append({'id': r['id'], 'text': r['text'], 'ref': r['text'], 'src': 'human'})
# Deterministic expansion to ~3000: paraphrase by fluff-drop / fluff-add (mirrors Compressor, no LLM needed)
base_texts = [q['text'] for q in queries]
i = 0
while len(queries) < 3000:
    t = base_texts[i % len(base_texts)]
    toks = t.split()
    if rng.random() < 0.5:
        toks = [w for w in toks if w.lower() not in set(FLUFF)]  # compress view
    else:
        toks = toks + [rng.choice(FLUFF)]  # fluff view
    queries.append({'id': f'aug-{i:04d}', 'text': ' '.join(toks), 'ref': base_texts[i % len(base_texts)], 'src': 'aug'})
    i += 1
print('queries:', len(queries))  # expect 3000

In [ ]:
# 2. FREE labeling: Groq 8B vs 70B (same key, OpenAI-compatible). Batched + checkpointed.
#   cheap   = llama-3.1-8b-instant
#   premium = llama-3.3-70b-versatile  (proxy=true, disclosed)
import httpx
CHEAP = 'llama-3.1-8b-instant'; PREMIUM = 'llama-3.3-70b-versatile'
BASE = 'https://api.groq.com/openai/v1'
H = {'Authorization': f'Bearer {GROQ_KEY}', 'Content-Type': 'application/json'}

def chat_once(model, text, max_tokens=128):
    for attempt in range(3):
        try:
            r = httpx.post(f'{BASE}/chat/completions', headers=H, timeout=30,
                json={'model': model, 'messages': [{'role':'user','content': text[:1500]}], 'temperature': 0.0, 'max_tokens': max_tokens})
            r.raise_for_status()
            j = r.json()
            return j['choices'][0]['message']['content'], j.get('usage', {})
        except Exception as e:
            time.sleep(2*(attempt+1))
            if attempt == 2: return f'[ERROR {e}]', {}

OUT = DRIVE/'groq_labels.jsonl'
done = set()
if OUT.exists():
    for l in open(OUT, encoding='utf-8'):
        try: done.add(json.loads(l)['id'])
        except Exception: pass
print('resume from', len(done))
fh = open(OUT, 'a', encoding='utf-8')
for q in queries:
    if q['id'] in done: continue
    c_text, c_use = chat_once(CHEAP, q['text'])
    p_text, p_use = chat_once(PREMIUM, q['text'])
    fh.write(json.dumps({'id': q['id'], 'text': q['text'], 'ref': q['ref'], 'src': q['src'],
        'cheap': c_text, 'premium': p_text, 'cheap_usage': c_use, 'premium_usage': p_use,
        'proxy': True, 'cheap_model': CHEAP, 'premium_model': PREMIUM})+'\n')
    fh.flush()
    time.sleep(0.4)  # stay inside Groq free rate limits
fh.close(); print('labeling done ->', OUT)

In [ ]:
# 3. Grade cheap_success LOCALLY (free): reward.py fidelity>=0.7 AND cheap~=premium -> success
# sys.path already set in Cell 0
from gateway.modules.m10_train.reward import reward
from gateway.modules.m4_router.difficulty import DifficultyScorer
from gateway.modules.m1_pipeline.hinglish import code_mix_ratio

scorer = DifficultyScorer()
def bucket(cm): return 'low' if cm < 0.25 else ('mid' if cm < 0.5 else 'high')

rows, human_pool = [], []
for l in open(DRIVE/'groq_labels.jsonl', encoding='utf-8'):
    r = json.loads(l)
    # fidelity of cheap vs ref, and cheap vs premium (agreement proxy)
    f_ref = reward(r['text'], r['text'], r['ref'], r['cheap'])
    agree = 1.0 if r['cheap'].strip().lower()[:60] == r['premium'].strip().lower()[:60] else 0.0
    ok = bool(f_ref >= 0.5 and (f_ref >= 0.7 or agree > 0))
    d = scorer.score(r['text']); cm = code_mix_ratio(r['text'])
    rows.append({'id': r['id'], 'nonconformity': round(d,6), 'cheap_success': ok,
        'synthetic': False, 'group': bucket(cm), 'fidelity': round(f_ref,4), 'src': r['src']})
print('graded', len(rows), 'success rate', round(sum(1 for x in rows if x['cheap_success'])/len(rows),3))

# Freeze splits 2000/500/500 stratified, seed 42. Test IDs sealed.
rng2 = random.Random(42)
by_g = {}
for x in rows: by_g.setdefault(x['group'], []).append(x)
train, calib, test = [], [], []
for g, lst in by_g.items():
    rng2.shuffle(lst)
    n = len(lst); n_te = max(1, int(0.17*n)); n_ca = max(1, int(0.17*n))
    test += lst[:n_te]; calib += lst[n_te:n_te+n_ca]; train += lst[n_te+n_ca:]
(DRIVE/'calibration_real.jsonl').write_text('\n'.join(json.dumps(x) for x in rows))
(DRIVE/'splits.json').write_text(json.dumps({'train': [x['id'] for x in train], 'calib': [x['id'] for x in calib], 'test': [x['id'] for x in test], 'seed': 42}, indent=2))
# human spot-check pool: 150 stratified
pool = (train[:50] + calib[:50] + test[:50])
import csv
with open(DRIVE/'human_check.csv','w',newline='',encoding='utf-8') as f:
    w = csv.DictWriter(f, fieldnames=['id','text','cheap','premium','auto_ok','human_ok']); w.writeheader()
    lab = {json.loads(l)['id']: json.loads(l) for l in open(DRIVE/'groq_labels.jsonl', encoding='utf-8')}
    for x in pool:
        r = lab[x['id']]; w.writerow({'id': x['id'], 'text': r['text'][:300], 'cheap': r['cheap'][:300], 'premium': r['premium'][:300], 'auto_ok': x['cheap_success'], 'human_ok': ''})
h = hashlib.sha256((DRIVE/'calibration_real.jsonl').read_bytes()).hexdigest()
(DRIVE/'DECISION_01.json').write_text(json.dumps({'stage':'01_label','calibration_sha256':h,'n':len(rows),'success_rate':round(sum(1 for x in rows if x['cheap_success'])/len(rows),4),'proxy':'llama-70b (free, disclosed)','human_check':'human_check.csv -> fill human_ok, re-upload before NB04'}, indent=2))
print('FROZEN calibration sha:', h)